In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_metadata_snapshot as ncbi_metadata_snapshot_module
import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.pca_kmeans as pca_kmeans_module
import src.pago_pipeline.pca_kmeans_snapshot as pca_kmeans_snapshot_module
import src.pago_pipeline.sweep_genes_snapshot as sweep_genes_snapshot_module
from src.pago_pipeline.storage import sha256_of_file

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_metadata_snapshot_module = importlib.reload(ncbi_metadata_snapshot_module)
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
pca_kmeans_module = importlib.reload(pca_kmeans_module)
pca_kmeans_snapshot_module = importlib.reload(pca_kmeans_snapshot_module)
sweep_genes_snapshot_module = importlib.reload(sweep_genes_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
load_latest_metadata_snapshot = ncbi_metadata_snapshot_module.load_latest_metadata_snapshot
load_latest_sweep_genes_snapshot = (
    sweep_genes_snapshot_module.load_latest_sweep_genes_snapshot
)
resolve_pca_kmeans_snapshot = pca_kmeans_snapshot_module.resolve_pca_kmeans_snapshot
latest_pca_kmeans_snapshot_is_available = (
    pca_kmeans_snapshot_module.latest_pca_kmeans_snapshot_is_available
)

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 — Define PCA/KMeans snapshot configuration
# =============================================================================

METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "ncbi" / "protein_metadata_csv"
)
SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "03-features" / "sweep_genes"
)
PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "04-analysis" / "pca_kmeans"
)

METADATA_SNAPSHOT_MODE = SnapshotMode.reuse_latest
SWEEP_GENES_SNAPSHOT_MODE = SnapshotMode.reuse_latest
PCA_KMEANS_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create

PCA_COMPONENT_COUNT_GRID = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 50, 100, 200)
KMEANS_CLUSTER_COUNT_GRID = tuple(range(2, 20))
PCA_SVD_SOLVER = "randomized"
PCA_RANDOM_STATE = 42
KMEANS_N_INIT = "auto"
SILHOUETTE_SAMPLE_SIZE = 8000
SILHOUETTE_RANDOM_STATE = 42
KMEANS_INITIALIZATION_REPEAT_COUNT = 6
SUBSAMPLE_REPEAT_COUNT = 6
SUBSAMPLE_FRACTION = 0.80
SUBSAMPLE_RANDOM_STATE = 123
MINIMUM_ACCEPTABLE_INIT_ARI_MIN = 0.70
MINIMUM_ACCEPTABLE_SUBSAMPLE_ARI_MIN = 0.60
EXPORT_PROJECTION_COMPONENT_COUNT = 3
UPDATE_LATEST_DIRECTORY = True

print(f"Metadata snapshot root directory: {METADATA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"SWeeP snapshot root directory: {SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA/KMeans output root directory: {PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA/KMeans snapshot mode: {PCA_KMEANS_SNAPSHOT_MODE}")
print(f"PCA component grid: {PCA_COMPONENT_COUNT_GRID}")
print(f"KMeans cluster grid: {KMEANS_CLUSTER_COUNT_GRID}")

Metadata snapshot root directory: C:\Programming\Python\pAgo-project\data\02-intermediate\ncbi\protein_metadata_csv
SWeeP snapshot root directory: C:\Programming\Python\pAgo-project\data\03-features\sweep_genes
PCA/KMeans output root directory: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans
PCA/KMeans snapshot mode: reuse_latest_or_create
PCA component grid: (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 50, 100, 200)
KMeans cluster grid: (2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19)


In [4]:
# =============================================================================
# CELL 4 — Resolve active source snapshots
# =============================================================================

metadata_snapshot_payload = load_latest_metadata_snapshot(
    snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
)
sweep_genes_snapshot_payload = load_latest_sweep_genes_snapshot(
    snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
)

metadata_snapshot_directory = metadata_snapshot_payload["snapshot_directory"]
metadata_manifest_file_path = metadata_snapshot_payload["manifest_file_path"]
metadata_csv_file_path = metadata_snapshot_payload["csv_file_path"]
metadata_manifest_payload = metadata_snapshot_payload["manifest"]

sweep_genes_snapshot_directory = sweep_genes_snapshot_payload["snapshot_directory"]
sweep_genes_manifest_file_path = sweep_genes_snapshot_payload["manifest_file_path"]
sweep_genes_manifest_payload = sweep_genes_snapshot_payload["manifest"]
sweep_genes_embeddings_file_path = sweep_genes_snapshot_payload["embeddings_file_path"]
sweep_genes_sequence_metadata_file_path = sweep_genes_snapshot_payload[
    "sequence_metadata_file_path"
]

print("Resolved source snapshots successfully.")
print(f"Metadata snapshot directory: {metadata_snapshot_directory}")
print(f"Metadata CSV path: {metadata_csv_file_path}")
print(f"SWeeP snapshot directory: {sweep_genes_snapshot_directory}")
print(f"SWeeP embeddings path: {sweep_genes_embeddings_file_path}")

Resolved source snapshots successfully.
Metadata snapshot directory: C:\Programming\Python\pAgo-project\data\02-intermediate\ncbi\protein_metadata_csv\latest
Metadata CSV path: C:\Programming\Python\pAgo-project\data\02-intermediate\ncbi\protein_metadata_csv\latest\protein_metadata.csv
SWeeP snapshot directory: C:\Programming\Python\pAgo-project\data\03-features\sweep_genes\latest
SWeeP embeddings path: C:\Programming\Python\pAgo-project\data\03-features\sweep_genes\latest\sweep_genes_embeddings_2800D.npy


In [5]:
# =============================================================================
# CELL 5 — Resolve active PCA/KMeans snapshot
# =============================================================================

pca_kmeans_snapshot_payload = resolve_pca_kmeans_snapshot(
    snapshot_mode=PCA_KMEANS_SNAPSHOT_MODE,
    snapshot_root_directory=PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY,
    source_sweep_snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    pca_component_count_grid=PCA_COMPONENT_COUNT_GRID,
    kmeans_cluster_count_grid=KMEANS_CLUSTER_COUNT_GRID,
    pca_svd_solver=PCA_SVD_SOLVER,
    pca_random_state=PCA_RANDOM_STATE,
    kmeans_n_init=KMEANS_N_INIT,
    silhouette_sample_size=SILHOUETTE_SAMPLE_SIZE,
    silhouette_random_state=SILHOUETTE_RANDOM_STATE,
    kmeans_initialization_repeat_count=KMEANS_INITIALIZATION_REPEAT_COUNT,
    subsample_repeat_count=SUBSAMPLE_REPEAT_COUNT,
    subsample_fraction=SUBSAMPLE_FRACTION,
    subsample_random_state=SUBSAMPLE_RANDOM_STATE,
    minimum_acceptable_init_ari_min=MINIMUM_ACCEPTABLE_INIT_ARI_MIN,
    minimum_acceptable_subsample_ari_min=MINIMUM_ACCEPTABLE_SUBSAMPLE_ARI_MIN,
    export_projection_component_count=EXPORT_PROJECTION_COMPONENT_COUNT,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

pca_kmeans_snapshot_directory = pca_kmeans_snapshot_payload["snapshot_directory"]
pca_kmeans_manifest_file_path = pca_kmeans_snapshot_payload["manifest_file_path"]
pca_kmeans_manifest_payload = pca_kmeans_snapshot_payload["manifest"]
pca_coordinates_file_path = pca_kmeans_snapshot_payload["pca_coordinates_file_path"]
explained_variance_ratio_file_path = pca_kmeans_snapshot_payload[
    "explained_variance_ratio_file_path"
]
cluster_assignments_file_path = pca_kmeans_snapshot_payload[
    "cluster_assignments_file_path"
]
stability_grid_file_path = pca_kmeans_snapshot_payload["stability_grid_file_path"]
profiling_log_file_path = pca_kmeans_snapshot_payload["profiling_log_file_path"]
alignment_report_file_path = pca_kmeans_snapshot_payload["alignment_report_file_path"]
pca_coordinates = pca_kmeans_snapshot_payload["pca_coordinates"]
explained_variance_ratio = pca_kmeans_snapshot_payload["explained_variance_ratio"]
cluster_assignments_dataframe = pca_kmeans_snapshot_payload["cluster_assignments"]
stability_grid_dataframe = pca_kmeans_snapshot_payload["stability_grid"]
profiling_log_dataframe = pca_kmeans_snapshot_payload["profiling_log"]
alignment_report = pca_kmeans_snapshot_payload["alignment_report"]

print("Resolved PCA/KMeans snapshot successfully.")
print(f"Snapshot directory: {pca_kmeans_snapshot_directory}")
print(f"PCA coordinates path: {pca_coordinates_file_path}")
print(f"Cluster assignments path: {cluster_assignments_file_path}")
print(f"Stability grid path: {stability_grid_file_path}")

m=  1, k= 2 | sil=0.6285 | initARI(min/mean)=0.985/0.992 | subARI(min/mean)=0.979/0.991
m=  1, k= 3 | sil=0.6048 | initARI(min/mean)=0.285/0.569 | subARI(min/mean)=0.278/0.613
m=  1, k= 4 | sil=0.5941 | initARI(min/mean)=0.976/0.988 | subARI(min/mean)=0.969/0.984
m=  1, k= 5 | sil=0.5234 | initARI(min/mean)=0.408/0.746 | subARI(min/mean)=0.403/0.618
m=  1, k= 6 | sil=0.5556 | initARI(min/mean)=0.624/0.852 | subARI(min/mean)=0.650/0.812
m=  1, k= 7 | sil=0.5526 | initARI(min/mean)=0.477/0.695 | subARI(min/mean)=0.456/0.630
m=  1, k= 8 | sil=0.5725 | initARI(min/mean)=0.853/0.942 | subARI(min/mean)=0.748/0.899
m=  1, k= 9 | sil=0.5621 | initARI(min/mean)=0.609/0.795 | subARI(min/mean)=0.537/0.759
m=  1, k=10 | sil=0.5324 | initARI(min/mean)=0.458/0.645 | subARI(min/mean)=0.541/0.705
m=  1, k=11 | sil=0.5347 | initARI(min/mean)=0.458/0.639 | subARI(min/mean)=0.510/0.636
m=  1, k=12 | sil=0.5431 | initARI(min/mean)=0.476/0.666 | subARI(min/mean)=0.465/0.640
m=  1, k=13 | sil=0.5397 | initA

C:\Programming\Python\pAgo-project\src\pago_pipeline\pca_kmeans_snapshot.py:958: DtypeWarning: Columns (0: gbseq__secondary_accessions__secondary_accn, 1: taxonomy__09, 2: taxonomy__10, 3: reference__consortium, 4: feature__cds__qual__db_xref, 5: feature__cds__qual__gene, 6: feature__cds__qual__gene_synonym, 7: feature__cds__qual__old_locus_tag, 8: feature__gene__interval__accession, 9: feature__gene__location, 10: feature__gene__qual__g_o_function, 11: feature__gene__qual__gene, 12: feature__gene__qual__gene_synonym, 13: feature__gene__qual__locus_tag, 14: feature__het__interval__accession, 15: feature__het__interval__point, 16: feature__het__location, 17: feature__het__qual__heterogen, 18: feature__non_std_res__interval__accession, 19: feature__non_std_res__interval__point, 20: feature__non_std_res__location, 21: feature__non_std_res__qual__non_std_residue, 22: feature__protein__qual__function, 23: feature__protein__qual__g_o_component, 24: feature__protein__qual__name, 25: feature__

In [6]:
# =============================================================================
# CELL 6 — Print PCA/KMeans snapshot summary
# =============================================================================

pca_kmeans_manifest_file_sha256 = sha256_of_file(
    input_file_path=pca_kmeans_manifest_file_path,
)
cluster_assignments_file_sha256 = sha256_of_file(
    input_file_path=cluster_assignments_file_path,
)
stability_grid_file_sha256 = sha256_of_file(
    input_file_path=stability_grid_file_path,
)
selected_configuration_summary_dataframe = pd.DataFrame(
    [
        {
            "selected_pca_component_count": pca_kmeans_manifest_payload[
                "selected_pca_component_count"
            ],
            "selected_cluster_count_k": pca_kmeans_manifest_payload[
                "selected_cluster_count_k"
            ],
            "selected_variance_explained_fraction": pca_kmeans_manifest_payload[
                "selected_variance_explained_fraction"
            ],
            "selected_silhouette_best_sampled": pca_kmeans_manifest_payload[
                "selected_silhouette_best_sampled"
            ],
            "selected_init_ari_min": pca_kmeans_manifest_payload[
                "selected_init_ari_min"
            ],
            "selected_subsample_ari_min": pca_kmeans_manifest_payload[
                "selected_subsample_ari_min"
            ],
            "selection_reason": pca_kmeans_manifest_payload["selection_reason"],
            "final_sampled_silhouette_value": pca_kmeans_manifest_payload[
                "final_sampled_silhouette_value"
            ],
        }
    ]
)

print("PCA/KMeans snapshot is ready.")
print(
    f"Snapshot created at UTC: {pca_kmeans_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Sequence count: {pca_kmeans_manifest_payload['sequence_count']}")
print(f"PCA coordinates shape: {pca_coordinates.shape}")
print(f"Explained variance vector length: {explained_variance_ratio.shape[0]}")
print(f"Cluster assignments rows: {len(cluster_assignments_dataframe)}")
print(f"Stability grid rows: {len(stability_grid_dataframe)}")
print(f"Alignment report: {alignment_report}")
print(f"Cluster assignments SHA-256: {cluster_assignments_file_sha256}")
print(f"Stability grid SHA-256: {stability_grid_file_sha256}")
print(f"Manifest SHA-256: {pca_kmeans_manifest_file_sha256}")

display(selected_configuration_summary_dataframe)

PCA/KMeans snapshot is ready.
Snapshot created at UTC: 2026-04-12T20:59:05Z
Sequence count: 41345
PCA coordinates shape: (41345, 2)
Explained variance vector length: 2
Cluster assignments rows: 41345
Stability grid rows: 252
Alignment report: {'duplicate_protein_uid_in_metadata_count': 0, 'duplicate_protein_uid_in_sequence_count': 0, 'matched_metadata_row_count': 41345, 'metadata_row_count': 41345, 'missing_metadata_row_count': 0, 'missing_protein_uid_in_sequence_count': 0, 'sequence_row_count': 41345}
Cluster assignments SHA-256: 4bd985a101553d0763776c9ce0ab01848473d33b34dcb57707fef0ab9e530a8d
Stability grid SHA-256: 02d47eecd1575918f7ab7fad5a7e6036fbf9574ac799bedcfa7d285a15931860
Manifest SHA-256: 50ae1317d969f5240c10d9662e41a0bcbdc1d9cfda5bfbe1dd53df2b54aae700


,selected_pca_component_count,selected_cluster_count_k,selected_variance_explained_fraction,selected_silhouette_best_sampled,selected_init_ari_min,selected_subsample_ari_min,selection_reason,final_sampled_silhouette_value
0,2,3,0.06573,0.655849,0.99849,0.993353,threshold_pass_then_max_silhouette,0.655849


In [7]:
# =============================================================================
# CELL 7 — Preview ranked configurations from the stability grid
# =============================================================================

stability_grid_ranked_dataframe = stability_grid_dataframe.sort_values(
    [
        "composite_score_silhouette_times_min_ari",
        "silhouette_best_sampled_filled",
        "variance_explained_fraction",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

print("Top ranked PCA/KMeans configurations:")
display(stability_grid_ranked_dataframe.head(10))

Top ranked PCA/KMeans configurations:


,pca_component_count,variance_explained_fraction,k,silhouette_best_sampled,init_ari_mean,init_ari_min,init_ari_max,init_ari_pair_count,subsample_ari_mean,subsample_ari_min,subsample_ari_max,subsample_ari_pair_count,subsample_fraction,subsample_point_count,mean_pairwise_intersection_size,silhouette_best_sampled_filled,init_ari_min_clipped,subsample_ari_min_clipped,composite_score_silhouette_times_min_ari
0,2,0.065730,3,0.655849,0.999295,0.998490,1.000000,15,0.997199,0.993353,0.999853,15,0.8,33076,26450.333333,0.655849,0.998490,0.993353,0.650505
1,1,0.041321,2,0.628539,0.991643,0.985462,0.999760,15,0.991451,0.978982,0.999624,15,0.8,33076,26450.333333,0.628539,0.985462,0.978982,0.606382
2,1,0.041321,4,0.594079,0.988096,0.976264,0.999122,15,0.983545,0.968883,0.997156,15,0.8,33076,26450.333333,0.594079,0.976264,0.968883,0.561931
3,3,0.079374,3,0.561681,0.999288,0.998275,1.000000,15,0.996475,0.992093,0.999663,15,0.8,33076,26450.333333,0.561681,0.998275,0.992093,0.556279
4,2,0.065730,4,0.499818,0.993281,0.988085,1.000000,15,0.991492,0.982440,0.997850,15,0.8,33076,26450.333333,0.499818,0.988085,0.982440,0.485190
5,1,0.041321,8,0.572478,0.941648,0.853045,0.999313,15,0.898771,0.748366,0.994908,15,0.8,33076,26450.333333,0.572478,0.853045,0.748366,0.365464
6,5,0.099001,7,0.382942,0.946666,0.844749,0.999389,15,0.992148,0.984732,0.997743,15,0.8,33076,26450.333333,0.382942,0.844749,0.984732,0.318551
7,3,0.079374,6,0.414110,0.996227,0.990200,1.000000,15,0.836104,0.735718,0.978641,15,0.8,33076,26450.333333,0.414110,0.990200,0.735718,0.301682
8,2,0.065730,7,0.484937,0.898257,0.811064,0.998852,15,0.911765,0.746285,0.996608,15,0.8,33076,26450.333333,0.484937,0.811064,0.746285,0.293525
9,4,0.090497,9,0.409804,0.910022,0.839804,0.990336,15,0.892468,0.801605,0.991896,15,0.8,33076,26450.333333,0.409804,0.839804,0.801605,0.275877


In [8]:
# =============================================================================
# CELL 8 — Preview cluster assignments and PCA outputs
# =============================================================================

cluster_assignment_preview_row_limit = 10
cluster_assignment_preview_dataframe = cluster_assignments_dataframe.head(
    cluster_assignment_preview_row_limit
).copy()
cluster_size_summary_dataframe = (
    cluster_assignments_dataframe["cluster_label"]
    .value_counts()
    .rename_axis("cluster_label")
    .reset_index(name="row_count")
    .sort_values("cluster_label")
    .reset_index(drop=True)
)
pca_coordinate_preview_dataframe = pd.DataFrame(
    np.asarray(pca_coordinates[:5]),
    columns=[f"pc{i + 1}" for i in range(pca_coordinates.shape[1])],
)

print("Cluster assignments preview:")
display(cluster_assignment_preview_dataframe)
print("Cluster size summary:")
display(cluster_size_summary_dataframe)
print("Selected PCA coordinate preview:")
display(pca_coordinate_preview_dataframe.iloc[:, : min(10, pca_coordinate_preview_dataframe.shape[1])])

Cluster assignments preview:


,sequence_index,record_id,description,sequence_length,protein_uid,gbseq__accession_version,gbseq__comment,gbseq__create_date,gbseq__definition,gbseq__division,...,feature__source__qual__serotype,feature__source__qual__serovar,feature__source__qual__specimen_voucher,feature__source__qual__strain,feature__source__qual__sub_species,feature__source__qual__sub_strain,feature__source__qual__type_material,cluster_label,pc1,pc2
0,0,protein_uid=1000250755|accession=KXK13845.1|le...,protein_uid=1000250755|accession=KXK13845.1|le...,709,1000250755,KXK13845.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ15_CFX003003232 [C...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,-0.648314,0.347258
1,1,protein_uid=1000266463|accession=KXK28958.1|le...,protein_uid=1000266463|accession=KXK28958.1|le...,1043,1000266463,KXK28958.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ01_02401 [Candidat...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2.243139,0.800604
2,2,protein_uid=1000285434|accession=KXK47085.1|le...,protein_uid=1000285434|accession=KXK47085.1|le...,365,1000285434,KXK47085.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ10_BCD003000691 [B...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1.997013,-0.723671
3,3,protein_uid=1000285973|accession=KXK47585.1|le...,protein_uid=1000285973|accession=KXK47585.1|le...,684,1000285973,KXK47585.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: Piwi domain-containing protein [Bacteroid...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2.116695,-0.236173
4,4,protein_uid=1000287044|accession=KXK48581.1|le...,protein_uid=1000287044|accession=KXK48581.1|le...,713,1000287044,KXK48581.1,##Genome-Assembly-Data-START## ; Assembly Date...,24-FEB-2016,MAG: hypothetical protein UZ13_03613 [Chlorofl...,ENV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,-1.701769,0.899859
5,5,protein_uid=1000371080|accession=WP_061113836....,protein_uid=1000371080|accession=WP_061113836....,191,1000371080,WP_061113836.1,"REFSEQ: This record represents a single, non-r...",24-FEB-2016,"pPIWI_RE module domain-containing protein, par...",BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3.170253,1.616078
6,6,protein_uid=1000882329|accession=WP_061139231....,protein_uid=1000882329|accession=WP_061139231....,560,1000882329,WP_061139231.1,"REFSEQ: This record represents a single, non-r...",25-FEB-2016,MULTISPECIES: RNaseH domain-containing protein...,BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,5.747983,6.130323
7,7,protein_uid=1000882353|accession=WP_061139255....,protein_uid=1000882353|accession=WP_061139255....,223,1000882353,WP_061139255.1,"REFSEQ: This record represents a single, non-r...",25-FEB-2016,MULTISPECIES: pPIWI_RE module domain-containin...,BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3.264804,1.901171
8,8,protein_uid=1000934900|accession=WP_061181381....,protein_uid=1000934900|accession=WP_061181381....,346,1000934900,WP_061181381.1,"REFSEQ: This record represents a single, non-r...",25-FEB-2016,restriction endonuclease-related protein [Pseu...,BCT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.580697,-0.664696
9,9,protein_uid=1001028003|accession=KXO00930.1|le...,protein_uid=1001028003|accession=KXO00930.1|le...,703,1001028003,KXO00930.1,Annotation was added by the NCBI Prokaryotic G...,25-FEB-2016,hypothetical protein LS48_00135 [Aequorivita a...,BCT,...,NaN,NaN,NaN,D-24,NaN,NaN,type strain of Vitellibacter aquimaris,0,2.376128,-0.559212


Cluster size summary:


,cluster_label,row_count
0,0,33503
1,1,6486
2,2,1356


Selected PCA coordinate preview:


,pc1,pc2
0,-0.648314,0.347258
1,2.243139,0.800604
2,1.997013,-0.723671
3,2.116695,-0.236173
4,-1.701769,0.899859


In [ ]:
# =============================================================================
# CELL 9 — Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- metadata_snapshot_directory")
print("- metadata_csv_file_path")
print("- sweep_genes_snapshot_directory")
print("- sweep_genes_embeddings_file_path")
print("- pca_kmeans_snapshot_directory")
print("- pca_coordinates_file_path")
print("- explained_variance_ratio_file_path")
print("- cluster_assignments_file_path")
print("- stability_grid_file_path")
print("- profiling_log_file_path")
print("- alignment_report_file_path")
print("- pca_kmeans_manifest_payload")
print("- pca_coordinates")
print("- explained_variance_ratio")
print("- cluster_assignments_dataframe")
print("- stability_grid_dataframe")
print("- stability_grid_ranked_dataframe")
print("- profiling_log_dataframe")
print("- alignment_report")
print("- selected_configuration_summary_dataframe")
print("- cluster_size_summary_dataframe")